# FieldVision Dataset EDA — Stage 1
## Dataset semantics, labels, modalities, and structure

**Purpose:** understand what each dataset actually contains before training any models.

This notebook is intentionally lightweight. It does **not** train neural networks.

It will help answer:
- What is one sample?
- What labels/annotations exist?
- What metadata or sensing streams exist?
- What image sizes/formats/channels are present?
- What tasks can each dataset realistically support?

## 0. Local setup

Recommended: run this notebook **locally** on the computer that already has `datasets.zip`.

The code below first looks for:
- `~/Downloads/datasets/`
- `~/Downloads/datasets.zip`

If only the ZIP exists, you can either extract it manually or use the optional extraction cell below.

In [7]:
from pathlib import Path
import os, random, json, zipfile
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import xml.etree.ElementTree as ET

HOME = Path.home()
PROJECT_ROOT = HOME / "Downloads" / "NOMS_Fieldvision"

# Your current layout:
# ~/Downloads/NOMS_Fieldvision/
#   datasets.zip
#   datasets/          <- created after extraction
#   notebooks/code

DATASETS_ZIP = PROJECT_ROOT / "datasets.zip"
DATASETS_DIR = PROJECT_ROOT / "datasets"

print("Project root:", PROJECT_ROOT)
print("datasets/ exists:", DATASETS_DIR.exists())
print("datasets.zip exists:", DATASETS_ZIP.exists())

if not DATASETS_DIR.exists():
    print(
        "\nThe extracted datasets folder was not found.\n"
        "Run the optional extraction cell below once."
    )


Project root: /Users/bwbgv/Downloads/NOMS_Fieldvision
datasets/ exists: True
datasets.zip exists: True


### Optional: extract `datasets.zip`

Only run this if you have enough free disk space.  
If the ZIP is very large, manual extraction may be easier to monitor.

In [8]:
# OPTIONAL — run only if DATASETS_DIR does not already exist.
# This extracts the archive inside ~/Downloads/NOMS_Fieldvision/.
# If the ZIP itself contains a top-level "datasets" directory,
# this produces ~/Downloads/NOMS_Fieldvision/datasets/.

if not DATASETS_DIR.exists():
    with zipfile.ZipFile(DATASETS_ZIP, "r") as z:
        z.extractall(PROJECT_ROOT)
    print("Extraction complete.")
else:
    print("datasets/ already exists; skipping extraction.")

print("Dataset search root:", DATASETS_DIR)


datasets/ already exists; skipping extraction.
Dataset search root: /Users/bwbgv/Downloads/NOMS_Fieldvision/datasets


## 1. Discover dataset folders automatically

In [9]:
if not DATASETS_DIR.exists():
    raise FileNotFoundError(
        f"{DATASETS_DIR} does not exist. Extract datasets.zip first."
    )

print("Searching recursively under:", DATASETS_DIR)

def find_dataset_folder(keyword):
    candidates = [
        p for p in DATASETS_DIR.rglob("*")
        if p.is_dir() and keyword.lower() in p.name.lower()
    ]
    # Prefer the shortest path below DATASETS_DIR to avoid accidentally
    # selecting a deeply nested folder with a similar name.
    if not candidates:
        return None
    candidates.sort(key=lambda p: len(p.parts))
    return candidates[0]

DATASETS = {
    "RumexWeeds": find_dataset_folder("rumex"),
    "LeafySpurge": find_dataset_folder("leafy"),
    "CWD30": find_dataset_folder("cwd"),
}

DATASETS = {k: v for k, v in DATASETS.items() if v is not None}

print("\nDetected datasets:")
for k, v in DATASETS.items():
    print(f"{k}: {v}")

if not DATASETS:
    raise RuntimeError(
        "No known dataset folders were detected. "
        "Inspect the extracted ZIP structure under DATASETS_DIR."
    )


Searching recursively under: /Users/bwbgv/Downloads/NOMS_Fieldvision/datasets

Detected datasets:
RumexWeeds: /Users/bwbgv/Downloads/NOMS_Fieldvision/datasets/datasets/RumexWeeds
LeafySpurge: /Users/bwbgv/Downloads/NOMS_Fieldvision/datasets/datasets/LeafySpurge
CWD30: /Users/bwbgv/Downloads/NOMS_Fieldvision/datasets/datasets/CWD30


## 2. Basic image collection helper

In [10]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}

def collect_images(root):
    return [
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

for name, path in DATASETS.items():
    imgs = collect_images(path)
    print(f"{name}: {len(imgs):,} images")

RumexWeeds: 5,510 images
LeafySpurge: 2,439 images
CWD30: 0 images


## 3. Lightweight image catalog

Samples at most 1,000 images per dataset.

In [11]:
CATALOG_COLUMNS = [
    "dataset",
    "relative_path",
    "filename",
    "extension",
    "width",
    "height",
    "megapixels",
    "channels_or_mode",
    "format",
    "file_size_mb",
    "error",
]

def build_image_catalog(dataset_name, dataset_path, max_images=1000, seed=42):
    images = collect_images(dataset_path)

    rng = random.Random(seed)
    if len(images) > max_images:
        images = rng.sample(images, max_images)

    rows = []

    for p in images:
        try:
            file_size_mb = p.stat().st_size / (1024 ** 2)

            with Image.open(p) as img:
                rows.append({
                    "dataset": dataset_name,
                    "relative_path": str(p.relative_to(dataset_path)),
                    "filename": p.name,
                    "extension": p.suffix.lower(),
                    "width": img.width,
                    "height": img.height,
                    "megapixels": img.width * img.height / 1e6,
                    "channels_or_mode": img.mode,
                    "format": img.format,
                    "file_size_mb": file_size_mb,
                    "error": None,
                })
        except Exception as e:
            rows.append({
                "dataset": dataset_name,
                "relative_path": str(p.relative_to(dataset_path)),
                "filename": p.name,
                "extension": p.suffix.lower(),
                "width": None,
                "height": None,
                "megapixels": None,
                "channels_or_mode": None,
                "format": None,
                "file_size_mb": None,
                "error": str(e),
            })

    # Important: keep the expected columns even when zero images are found.
    return pd.DataFrame(rows, columns=CATALOG_COLUMNS)


catalogs = {}

for name, path in DATASETS.items():
    print(f"\nProcessing {name}...")
    catalogs[name] = build_image_catalog(name, path, max_images=1000)

    if catalogs[name].empty:
        print(
            f"No directly readable images found for {name}. "
            "This is expected if the dataset is still stored inside a ZIP."
        )
        continue

    display(catalogs[name].head())

    valid = catalogs[name].dropna(subset=["width"])

    if len(valid):
        display(
            valid[
                ["file_size_mb", "width", "height", "megapixels"]
            ].describe()
        )
    else:
        print(f"No readable image metadata found for {name}.")



Processing RumexWeeds...


,dataset,relative_path,filename,extension,width,height,megapixels,channels_or_mode,format,file_size_mb,error
0,RumexWeeds,20210807_lundholm/seq10/imgs/20210807_lundholm...,20210807_lundholm_rgb_10_1628331855465779152.png,.png,1920,1200,2.304,RGB,PNG,5.440947,None
1,RumexWeeds,20210806_stengard/seq19/imgs/20210806_stengard...,20210806_stengard_rgb_19_1628243594019753431.png,.png,1920,1200,2.304,RGB,PNG,5.301497,None
2,RumexWeeds,20210806_stengard/seq14/imgs/20210806_stengard...,20210806_stengard_rgb_14_1628243420021427673.png,.png,1920,1200,2.304,RGB,PNG,5.162028,None
3,RumexWeeds,20210908_lundholm/seq13/imgs/20210908_lundholm...,20210908_lundholm_rgb_13_1631093047450103234.png,.png,1920,1200,2.304,RGB,PNG,5.139147,None
4,RumexWeeds,20210806_hegnstrup/seq9/imgs/20210806_hegnstru...,20210806_hegnstrup_rgb_9_1628251273719851730.png,.png,1920,1200,2.304,RGB,PNG,4.900403,None


,file_size_mb,width,height,megapixels
count,1000.000000,1000.0,1000.0,1000.000
mean,5.219250,1920.0,1200.0,2.304
std,0.142754,0.0,0.0,0.000
min,4.762534,1920.0,1200.0,2.304
25%,5.127532,1920.0,1200.0,2.304
50%,5.205079,1920.0,1200.0,2.304
75%,5.318081,1920.0,1200.0,2.304
max,5.571625,1920.0,1200.0,2.304



Processing LeafySpurge...


,dataset,relative_path,filename,extension,width,height,megapixels,channels_or_mode,format,file_size_mb,error
0,LeafySpurge,data/orthomosaic/orthomosaic_30_20.tif,orthomosaic_30_20.tif,.tif,4096,4096,16.777216,RGBA,TIFF,32.861188,None
1,LeafySpurge,data/orthomosaic/orthomosaic_21_18.tif,orthomosaic_21_18.tif,.tif,4096,4096,16.777216,RGBA,TIFF,31.144897,None
2,LeafySpurge,data/crop/train/absent/854_8.tif,854_8.tif,.tif,39,39,0.001521,RGBA,TIFF,0.006172,None
3,LeafySpurge,data/crop/train/present/824_8.tif,824_8.tif,.tif,39,39,0.001521,RGBA,TIFF,0.006172,None
4,LeafySpurge,data/crop/train/present/778_7.tif,778_7.tif,.tif,39,39,0.001521,RGBA,TIFF,0.006172,None


,file_size_mb,width,height,megapixels
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,8.603167,1493.558000,1492.948000,4.962491
std,11.621860,1653.890708,1654.003505,7.275636
min,0.002918,39.000000,39.000000,0.001521
25%,0.006172,39.000000,39.000000,0.001521
50%,4.000870,1024.000000,1024.000000,1.048576
75%,16.623381,4096.000000,4096.000000,16.777216
max,35.108571,4096.000000,4096.000000,16.777216



Processing CWD30...
No directly readable images found for CWD30. This is expected if the dataset is still stored inside a ZIP.


## 4. RumexWeeds: inspect JSON/XML metadata

This checks what GPS, IMU, odometry, RGB metadata, and annotations actually contain.

In [12]:
if "RumexWeeds" in DATASETS:
    rumex_path = DATASETS["RumexWeeds"]

    json_files = list(rumex_path.rglob("*.json"))
    xml_files = list(rumex_path.rglob("*.xml"))

    print("JSON files:", len(json_files))
    print("XML files :", len(xml_files))

    print("\nExample JSON files:")
    for p in json_files[:10]:
        print(p.relative_to(rumex_path))

    print("\nExample XML files:")
    for p in xml_files[:10]:
        print(p.relative_to(rumex_path))
else:
    json_files, xml_files = [], []

JSON files: 397
XML files : 196

Example JSON files:
20210807_lundholm/metadata.json
20211006_stengard/metadata.json
20210908_lundholm/metadata.json
20210806_hegnstrup/metadata.json
20210806_stengard/metadata.json
20210806_stengard/seq8/odom.json
20210806_stengard/seq8/rgb.json
20210806_stengard/seq8/gps.json
20210806_stengard/seq8/imu.json
20210806_stengard/seq1/odom.json

Example XML files:
20210806_stengard/seq8/annotations.xml
20210806_stengard/seq8/annotations_seg.xml
20210806_stengard/seq1/annotations.xml
20210806_stengard/seq1/annotations_seg.xml
20210806_stengard/seq6/annotations.xml
20210806_stengard/seq6/annotations_seg.xml
20210806_stengard/seq7/annotations.xml
20210806_stengard/seq7/annotations_seg.xml
20210806_stengard/seq0/annotations.xml
20210806_stengard/seq0/annotations_seg.xml


In [13]:
def summarize_json(path):
    print("\n" + "=" * 80)
    print(path.name)
    print("=" * 80)

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print("Top-level type:", type(data).__name__)

        if isinstance(data, dict):
            print("Keys:", list(data.keys())[:30])
            for key, value in list(data.items())[:10]:
                if isinstance(value, list):
                    print(f"{key}: list with {len(value)} items")
                    if value:
                        print(" first item:", str(value[0])[:300])
                elif isinstance(value, dict):
                    print(f"{key}: dict keys =", list(value.keys())[:20])
                else:
                    print(f"{key}: {str(value)[:200]}")
        elif isinstance(data, list):
            print("Number of records:", len(data))
            if data:
                print("First record:", str(data[0])[:1000])

    except Exception as e:
        print("Could not read:", e)

if json_files:
    for keyword in ["gps", "imu", "odom", "rgb"]:
        matches = [p for p in json_files if keyword in p.name.lower()]
        print("\n" + "#" * 80)
        print(keyword.upper(), "FILES:", len(matches))
        for p in matches[:2]:
            summarize_json(p)


################################################################################
GPS FILES: 98

gps.json
Top-level type: dict
Keys: ['20210806_stengard_8_1628241847819304759', '20210806_stengard_8_1628241824218985816', '20210806_stengard_8_1628241846015056295', '20210806_stengard_8_1628241837818401994', '20210806_stengard_8_1628241820017949861', '20210806_stengard_8_1628241838818334571', '20210806_stengard_8_1628241837220126162', '20210806_stengard_8_1628241867617930042', '20210806_stengard_8_1628241838018126231', '20210806_stengard_8_1628241830818793602', '20210806_stengard_8_1628241843618409247', '20210806_stengard_8_1628241835614794434', '20210806_stengard_8_1628241834419144636', '20210806_stengard_8_1628241839817556699', '20210806_stengard_8_1628241834220059760', '20210806_stengard_8_1628241837017614081', '20210806_stengard_8_1628241831619293797', '20210806_stengard_8_1628241830018568506', '20210806_stengard_8_1628241849419233553', '20210806_stengard_8_1628241845418344893', '20210

In [14]:
def inspect_xml_structure(path, max_elements=5000):
    try:
        tree = ET.parse(path)
        root = tree.getroot()
        tags = Counter()

        for i, elem in enumerate(root.iter()):
            tags[elem.tag] += 1
            if i >= max_elements:
                break

        print("\nFile:", path.name)
        print("Root:", root.tag)
        display(pd.DataFrame(tags.most_common(30), columns=["tag", "count"]))

    except Exception as e:
        print("Error:", e)

def show_xml_examples(path, n=20):
    tree = ET.parse(path)
    root = tree.getroot()
    shown = 0

    for elem in root.iter():
        if elem.attrib:
            print(
                "TAG:", elem.tag,
                "| ATTR:", elem.attrib,
                "| TEXT:", (elem.text or "").strip()[:100]
            )
            shown += 1
            if shown >= n:
                break

if xml_files:
    for p in xml_files[:2]:
        inspect_xml_structure(p)

    print("\nExample annotated elements:")
    show_xml_examples(xml_files[0])


File: annotations.xml
Root: annotations


,tag,count
0,box,1106
1,ellipse,1073
2,image,250
3,label,2
4,name,2
5,attributes,2
6,annotations,1
7,version,1
8,meta,1
9,task,1



File: annotations_seg.xml
Root: annotations


,tag,count
0,polygon,2331
1,image,250
2,label,2
3,name,2
4,attributes,2
5,annotations,1
6,version,1
7,meta,1
8,task,1
9,flipped,1



Example annotated elements:
TAG: image | ATTR: {'name': '20210806_stengard_rgb_8_1628241822819154684.png', 'id': '0', 'width': '1920', 'height': '1200'} | TEXT: 
TAG: box | ATTR: {'label': 'rumex_obtusifolius', 'xtl': '1641', 'ytl': '812', 'xbr': '1805', 'ybr': '1000', 'rotation': '0.0', 'occluded': '0', 'z_order': '1', 'source': 'manual'} | TEXT: 
TAG: box | ATTR: {'label': 'rumex_obtusifolius', 'xtl': '923', 'ytl': '624', 'xbr': '1130', 'ybr': '771', 'rotation': '0.0', 'occluded': '0', 'z_order': '1', 'source': 'manual'} | TEXT: 
TAG: ellipse | ATTR: {'label': 'rumex_obtusifolius', 'cx': '1694', 'cy': '889', 'rx': '35', 'ry': '33', 'occluded': '0', 'z_order': '1', 'source': 'manual'} | TEXT: 
TAG: ellipse | ATTR: {'label': 'rumex_obtusifolius', 'cx': '1004', 'cy': '690', 'rx': '35', 'ry': '36', 'occluded': '0', 'z_order': '1', 'source': 'manual'} | TEXT: 
TAG: image | ATTR: {'name': '20210806_stengard_rgb_8_1628241823219784797.png', 'id': '0', 'width': '1920', 'height': '1200'} | TE

## 5. LeafySpurge: infer sample semantics from folder structure

In [15]:
if "LeafySpurge" in DATASETS:
    leafy_path = DATASETS["LeafySpurge"]
    leafy_images = collect_images(leafy_path)

    rows = []
    for p in leafy_images:
        rel = str(p.relative_to(leafy_path)).lower()

        if "orthomosaic" in rel:
            subset_type = "orthomosaic"
        elif "context" in rel:
            subset_type = "context"
        elif "crop" in rel:
            subset_type = "crop"
        else:
            subset_type = "other"

        if "present" in rel:
            label = "present"
        elif "absent" in rel:
            label = "absent"
        else:
            label = "unknown"

        rows.append({
            "path": str(p),
            "relative_path": str(p.relative_to(leafy_path)),
            "subset_type": subset_type,
            "label": label
        })

    leafy_semantics = pd.DataFrame(rows)

    display(
        leafy_semantics.groupby(
            ["subset_type", "label"]
        ).size().reset_index(name="count")
    )
else:
    leafy_semantics = pd.DataFrame()

,subset_type,label,count
0,context,absent,443
1,context,present,457
2,crop,absent,443
3,crop,present,457
4,orthomosaic,unknown,634
5,other,unknown,5


## 6. Inspect TIFF channel structure

This does **not** assume TIFF means multispectral.

In [16]:
if "LeafySpurge" in DATASETS:
    tif_files = [
        p for p in leafy_images
        if p.suffix.lower() in {".tif", ".tiff"}
    ]

    print("TIFF files:", len(tif_files))

    for p in tif_files[:20]:
        try:
            with Image.open(p) as img:
                print(
                    p.name,
                    "| size =", img.size,
                    "| mode =", img.mode,
                    "| bands =", img.getbands(),
                    "| frames =", getattr(img, "n_frames", 1)
                )
        except Exception as e:
            print("ERROR:", p.name, e)

TIFF files: 2434
orthomosaic_26_17.tif | size = (4096, 4096) | mode = RGBA | bands = ('R', 'G', 'B', 'A') | frames = 1
orthomosaic_26_03.tif | size = (4096, 4096) | mode = RGBA | bands = ('R', 'G', 'B', 'A') | frames = 1
orthomosaic_28_06.tif | size = (4096, 4096) | mode = RGBA | bands = ('R', 'G', 'B', 'A') | frames = 1
orthomosaic_21_22.tif | size = (4096, 4096) | mode = RGBA | bands = ('R', 'G', 'B', 'A') | frames = 1
orthomosaic_28_12.tif | size = (4096, 4096) | mode = RGBA | bands = ('R', 'G', 'B', 'A') | frames = 1
orthomosaic_09_23.tif | size = (4096, 4096) | mode = RGBA | bands = ('R', 'G', 'B', 'A') | frames = 1
orthomosaic_31_27.tif | size = (4096, 4096) | mode = RGBA | bands = ('R', 'G', 'B', 'A') | frames = 1
orthomosaic_19_26.tif | size = (4096, 4096) | mode = RGBA | bands = ('R', 'G', 'B', 'A') | frames = 1
orthomosaic_10_16.tif | size = (4096, 4096) | mode = RGBA | bands = ('R', 'G', 'B', 'A') | frames = 1
orthomosaic_17_23.tif | size = (4096, 4096) | mode = RGBA | bands

## 7. CWD30: inspect ZIP without extracting it

If your CWD30 folder contains a large ZIP, this lets us inspect structure cheaply.

In [17]:
if "CWD30" in DATASETS:
    cwd_path = DATASETS["CWD30"]

    # Search recursively because CWD30 may contain data.zip.
    zip_files = list(cwd_path.rglob("*.zip"))

    print("ZIP files:")
    for p in zip_files:
        print(" -", p)

    if zip_files:
        zip_path = zip_files[0]

        with zipfile.ZipFile(zip_path, "r") as z:
            names = z.namelist()

            print("\nFiles inside ZIP:", len(names))

            extensions = Counter(
                Path(name).suffix.lower()
                for name in names
                if not name.endswith("/")
            )

            display(
                pd.DataFrame(
                    extensions.most_common(20),
                    columns=["extension", "count"]
                )
            )

            print("\nFirst 50 paths:")
            for name in names[:50]:
                print(name)
    else:
        print("No ZIP found inside the detected CWD30 directory.")


ZIP files:
 - /Users/bwbgv/Downloads/NOMS_Fieldvision/datasets/datasets/CWD30/data.zip

Files inside ZIP: 196306


,extension,count
0,.jpg,196275



First 50 paths:
data/
data/asian-flatsedge/
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162661.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162662.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162663.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162664.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162665.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162666.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162667.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162668.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162669.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162670.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162671.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162672.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162673.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162674.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162675.jpg
data/asian-flatsedge/weed_asian-flatsedge_w0_0_0162676.jpg
data/asian-

## 8. Create one dataset summary table

In [18]:
dataset_report = []

for dataset_name, df in catalogs.items():
    valid = df.dropna(subset=["width", "height"])

    if not len(valid):
        continue

    dataset_report.append({
        "dataset": dataset_name,
        "sampled_images": len(valid),
        "median_file_MB": valid["file_size_mb"].median(),
        "max_file_MB": valid["file_size_mb"].max(),
        "median_MPixels": valid["megapixels"].median(),
        "max_MPixels": valid["megapixels"].max(),
        "formats": ", ".join(valid["format"].dropna().astype(str).unique()),
        "modes": ", ".join(valid["channels_or_mode"].dropna().astype(str).unique())
    })

dataset_report_df = pd.DataFrame(dataset_report)
display(dataset_report_df)

,dataset,sampled_images,median_file_MB,max_file_MB,median_MPixels,max_MPixels,formats,modes
0,RumexWeeds,1000,5.205079,5.571625,2.304000,2.304000,PNG,RGB
1,LeafySpurge,1000,4.000870,35.108571,1.048576,16.777216,"TIFF, PNG","RGBA, RGB"


## 9. Save outputs

In [19]:
OUTPUT_DIR = PROJECT_ROOT / "eda_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

dataset_report_df.to_csv(
    OUTPUT_DIR / "dataset_summary.csv",
    index=False
)

for name, df in catalogs.items():
    df.to_csv(
        OUTPUT_DIR / f"{name}_image_catalog.csv",
        index=False
    )

if len(leafy_semantics):
    leafy_semantics.to_csv(
        OUTPUT_DIR / "leafyspurge_semantics.csv",
        index=False
    )

print("Saved outputs to:", OUTPUT_DIR)


Saved outputs to: /Users/bwbgv/Downloads/NOMS_Fieldvision/eda_outputs


## What to send back after running this notebook

The most useful outputs are:
- `dataset_summary.csv`
- `RumexWeeds_image_catalog.csv`
- `LeafySpurge_image_catalog.csv`
- `leafyspurge_semantics.csv`
- screenshots/text output from the Rumex JSON/XML inspection
- first-level CWD30 ZIP folder structure

Those are enough to design the next lightweight loaders correctly.